# Week 7 Assignment: Price Band Classification with Llama-3.2-3B (QLoRA)
---
This notebook implements a full pipeline to fine-tune Llama-3.2-3B (with QLoRA) for price band classification (budget, midrange, premium) instead of exact price prediction.

**Outline:**
1. Setup Dependencies and Experiment Configuration
2. Load and Analyze ed-donner Dataset
3. Create Dynamic Price Bands
4. Transform Labels to Price Classes
5. Reformat Prompts for Llama Chat Format
6. Configure QLoRA and Training Arguments
7. Define Utility Functions
8. Fine-Tune Llama-3.2-3B with QLoRA
9. Save Model and Tokenizer
10. Push Model to Hugging Face Hub
11. Evaluate Model Performance
12. Error Analysis of Misclassified Samples
---

## 1. Setup Dependencies and Experiment Configuration
Install and import required libraries for Hugging Face Transformers, PEFT, QLoRA, and experiment configuration.

In [ ]:
# Install required packages (uncomment if running in a new environment)
# !pip install transformers datasets peft accelerate bitsandbytes scikit-learn matplotlib seaborn
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, Dataset
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch
import random
import warnings
warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## 2. Load and Analyze ed-donner Dataset
Load the ed-donner dataset, inspect price distribution, and visualize statistics.

In [ ]:
# Utility function: Load dataset (CSV/XLSX/Parquet)
def load_price_data(file_path):
    ext = os.path.splitext(file_path)[-1].lower()
    if ext == '.csv':
        df = pd.read_csv(file_path)
    elif ext in ['.xlsx', '.xls']:
        df = pd.read_excel(file_path)
    elif ext == '.parquet':
        df = pd.read_parquet(file_path)
    else:
        raise ValueError(f"Unsupported file format: {ext}")
    return df

# Example usage (replace with actual file path)
df = load_price_data('ed_donner_products.csv')
df.head()

In [ ]:
# Utility function: Visualize price distribution and class balance
def plot_price_distribution(df, price_col):
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(df[price_col], bins=30, ax=ax)
    ax.set_title('Price Distribution')
    plt.show()

# Example usage
# plot_price_distribution(df, 'price')

## 3. Create Dynamic Price Bands
Calculate percentile thresholds and assign price bands (budget, midrange, premium) based on price distribution.

In [ ]:
# Utility function: Create price bands using percentiles
def create_price_bands(df, price_col, low_pct=0.33, high_pct=0.66):
    low = df[price_col].quantile(low_pct)
    high = df[price_col].quantile(high_pct)
    def band(price):
        if price <= low: return 'budget'
        elif price <= high: return 'midrange'
        else: return 'premium'
    df['price_band'] = df[price_col].apply(band)
    return df, low, high

# Example usage
# df, low, high = create_price_bands(df, 'price')
# df['price_band'].value_counts()

## 4. Transform Labels to Price Classes
Convert numeric price labels to categorical price classes for classification.

In [ ]:
# Utility function: Encode price bands as class labels
def encode_price_classes(df, band_col='price_band'):
    class_map = {'budget': 0, 'midrange': 1, 'premium': 2}
    df['price_class'] = df[band_col].map(class_map)
    return df, class_map

# Example usage
# df, class_map = encode_price_classes(df)
# df[['price_band', 'price_class']].head()

## 5. Reformat Prompts for Llama Chat Format
Prepare prompts for Llama-3.2-3B with one-word class output in chat format.

In [ ]:
# Utility function: Format prompts for Llama chat
def format_llama_prompts(df, text_col, band_col='price_band'):
    prompts = []
    for _, row in df.iterrows():
        prompt = {
            'messages': [
                {'role': 'system', 'content': 'Classify the product into price band: budget, midrange, or premium.'},
                {'role': 'user', 'content': str(row[text_col])},
                {'role': 'assistant', 'content': str(row[band_col])}
            ]
        }
        prompts.append(prompt)
    return prompts

# Example usage
# prompts = format_llama_prompts(df, 'description')
# prompts[0]

## 6. Configure QLoRA and Training Arguments
Set up quantization, LoRA adapters, and SFT training arguments for fine-tuning.

In [ ]:
# Utility function: Setup QLoRA and training arguments
def setup_qlora_and_training(model_name, lora_r=8, lora_alpha=16, lora_dropout=0.05, quantization_bits=4):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map='auto',
        torch_dtype=torch.float16,
        load_in_4bit=True,
        quantization_config={
            'load_in_4bit': True,
            'bnb_4bit_compute_dtype': torch.float16,
            'bnb_4bit_use_double_quant': True,
            'bnb_4bit_quant_type': 'nf4'
        }
    )
    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        bias='none',
        task_type='CAUSAL_LM'
    )
    model = get_peft_model(model, lora_config)
    return model, tokenizer, lora_config

# Example usage
# model, tokenizer, lora_config = setup_qlora_and_training('meta-llama/Llama-3-3B')

## 7. Define Utility Functions
Create reusable functions for data processing, evaluation, and error analysis.

In [ ]:
# Utility function: Prepare dataset for SFTTrainer
def prepare_sft_dataset(prompts):
    data = []
    for item in prompts:
        user_msg = item['messages'][1]['content']
        label = item['messages'][2]['content']
        data.append({'prompt': user_msg, 'label': label})
    return Dataset.from_pandas(pd.DataFrame(data))

# Utility function: Compute metrics for evaluation
def compute_metrics(pred):
    y_true = pred.label_ids
    y_pred = pred.predictions.argmax(axis=-1)
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, average='weighted'),
        'report': classification_report(y_true, y_pred, output_dict=True)
    }

## 8. Fine-Tune Llama-3.2-3B with QLoRA
Train the model using SFTTrainer with QLoRA configuration.

In [ ]:
# Utility function: Fine-tune Llama-3.2-3B with QLoRA
def train_llama_qlora(model, tokenizer, train_dataset, eval_dataset, output_dir='qlora_output', epochs=3, batch_size=8):
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        logging_strategy='epoch',
        report_to='none',
        fp16=True,
        push_to_hub=False
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )
    trainer.train()
    return trainer

## 9. Save Model and Tokenizer
Save the fine-tuned model and tokenizer locally.

In [ ]:
# Utility function: Save model and tokenizer
def save_model_and_tokenizer(trainer, output_dir='qlora_output'):
    trainer.model.save_pretrained(output_dir)
    trainer.tokenizer.save_pretrained(output_dir)
    print(f"Model and tokenizer saved to {output_dir}")

## 10. Push Model to Hugging Face Hub
Upload the trained model and tokenizer to Hugging Face Hub.

In [ ]:
# Utility function: Push model to Hugging Face Hub
def push_to_hf_hub(trainer, repo_id, output_dir='qlora_output', use_auth_token=True):
    trainer.model.push_to_hub(repo_id, use_auth_token=use_auth_token)
    trainer.tokenizer.push_to_hub(repo_id, use_auth_token=use_auth_token)
    print(f"Model and tokenizer pushed to Hugging Face Hub repo: {repo_id}")

## 11. Evaluate Model Performance
Evaluate the model on test data using accuracy, F1 score, precision, recall, classification report, and confusion matrix.

In [ ]:
# Utility function: Evaluate classification performance
def evaluate_classification(y_true, y_pred, labels=('budget', 'midrange', 'premium')):
    print('Classification Report:')
    print(classification_report(y_true, y_pred, labels=labels))
    print('Confusion Matrix:')
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()
    f1 = f1_score(y_true, y_pred, average=None, labels=labels)
    plt.bar(labels, f1)
    plt.title('Per-class F1 Score')
    plt.ylabel('F1 Score')
    plt.show()
    acc = accuracy_score(y_true, y_pred)
    print(f'Accuracy: {acc:.4f}')
    return cm, f1, acc

## 12. Error Analysis of Misclassified Samples
Analyze misclassified samples and visualize error patterns.

In [ ]:
# Utility function: Error analysis for misclassified samples
def error_analysis(df, y_true, y_pred, text_col='description', band_col='price_band'):
    errors = df.loc[y_true != y_pred]
    print(f"Number of misclassified samples: {len(errors)}")
    print(errors[[text_col, band_col]])
    return errors